# Semaine 9 — Jour 6 : Mémoire d’agents : isolation, durée et invalidation

Distinguer préférence de session, état d’agent et connaissance documentaire.

Parcours essentiel : 60 minutes (cours 20 min, lab 25 min, exercice 10 min, synthèse 5 min). Prévoir 2 à 3 heures pour l’unité complète avec entretien et challenge. Les modèles et services optionnels ajoutent un temps d’installation distinct.

Prérequis : Python 3.11+, JSON, notions de vecteurs et de persistance. Le parcours obligatoire utilise uniquement la bibliothèque standard et ne contacte aucun service externe.

Lire [les objectifs](learning_objectives.md), [le chapitre](chapter.md), puis [le lab](labs/README.md). Réaliser [les exercices](exercises.md), [l’entretien](interview.md) et [le challenge](challenge.md) avant les corrigés locaux.

Notebooks depuis la racine : `notebooks/week09/S9_J6_memoire_agents.ipynb` et `notebooks/week09/teacher/S9_J6_memoire_agents_teacher.ipynb`.

Critère de réussite : Invalider une note dérivée quand sa source change ou est supprimée.

Les calculs de retrieval et de persistance sont exécutables; le corpus est fictif. L’encodeur de base est lexical et la réponse finale est extractive. Les extensions neuronales sont séparées et n’ont pas été exécutées ici.


# Objectifs mesurables

- Distinguer préférence de session, état d’agent et connaissance documentaire.
- Isoler les notes par tenant, utilisateur et session avec un TTL explicite.
- Invalider une note dérivée quand sa source change ou est supprimée.

## Preuves attendues

Une sortie reproductible, un contrôle d’invariant et une limite du mécanisme correctement expliquée.


# S9 J6 — Mémoire d’agents : isolation, durée et invalidation


## 1. Ce jour prolonge les fondamentaux de mémoire
S2 présentait les types de mémoire et S3 le partage d’état. Ici l’objectif est opérationnel : que devient une information mémorisée quand la source change, quand la session se termine ou quand un autre utilisateur interroge le système ? La recherche documentaire et la mémoire d’agent ne doivent pas se transformer en un même sac de textes sans politique de cycle de vie.

Le store possède une table memory distincte. Une note peut représenter une préférence de forme ou une synthèse dérivée d’un document. Les notes ne sont jamais utilisées comme preuves factuelles dans la réponse extractive. Les citations finales proviennent exclusivement des chunks courants et autorisés.

## 2. Définir la clé de portée
Une note est identifiée par tenant, user_id, session_id et note_id. Connaître le même note_id dans une autre session ne donne pas accès au contenu. Le code reçoit ces champs d’un contexte supposé fiable; il ne fournit pas d’authentification. Un raccord réel dériverait tenant et utilisateur d’une identité vérifiée.

Le lab ne crée pas de mémoire globale ni de profil personnel implicite. Une note de style pour Alice dans s1 n’apparaît ni chez Bob dans s1, ni chez Alice dans s2. Étendre une préférence à toutes les sessions serait une décision de produit et de consentement à définir séparément.

## 3. TTL et horloge
remember enregistre expires_at = now + ttl. recall ne retourne que les notes dont expires_at est strictement supérieur à now. Une note expire donc exactement à l’échéance. Les tests injectent l’horloge pour éviter une attente réelle et rendre les frontières reproductibles.

L’expiration est logique : une ligne expirée peut encore se trouver dans le fichier SQLite. Le module n’exécute pas de purge automatique. forget supprime une note du store courant; sauvegardes et autres copies nécessiteraient une politique distincte. Ne pas confondre invisibilité à la lecture et effacement de toutes les copies physiques.

## 4. Mémoire liée à une source
Une note peut référencer doc_id. Le store conserve alors la version courante au moment de l’écriture. Lors de recall, une jointure vérifie que la source possède encore cette version. Une mise à jour masque donc une ancienne synthèse au lieu de la présenter comme un savoir récent.

La suppression documentaire retire aussi les notes référencées dans le tenant. Réingérer plus tard un document de même nom ne ressuscite pas ces notes supprimées. Une note sans source reste une note de session; elle ne reçoit pas artificiellement un statut de vérité documentaire.

## 5. Empoisonnement et corrections
Un texte reçu peut demander de mémoriser une fausse préférence, une autorisation ou un secret. La présence dans un échange ne suffit pas à autoriser l’écriture en mémoire. L’application doit distinguer contenu à utiliser une fois et fait durable à conserver. Le lab expose une API explicite, sans extraction automatique de « faits » par un modèle.

Une correction peut remplacer la note de même clé; un oubli la supprime. Il n’y a pas d’historique des anciennes notes ni de journal d’approbation dans cette table. Ces extensions peuvent être utiles, mais ne sont pas présentées comme déjà disponibles. Une note qui prétend modifier les droits ne change aucun filtre de recherche.

## 6. Expérience du jour
Créer une préférence, tester les trois axes d’isolation, puis avancer l’horloge jusqu’à la date exacte d’expiration. Créer ensuite une note liée à une source et modifier cette source. Vérifier qu’aucune réponse documentaire ne cite la note comme autorité. Cette séparation évite de transformer une mémoire commode en vecteur de fuite ou de connaissance périmée.

## À retenir
Une mémoire exploitable a une portée, une durée, une provenance et une règle d’invalidation. Sa récupération ne lui confère pas automatiquement une valeur de preuve.


## Diagramme du jour

```mermaid
flowchart TD
 A["Note explicitement écrite"] --> B["Tenant, utilisateur, session"]
 B --> C["Date d’expiration"]
 C --> D{"Source encore courante ?"}
 D -->|oui ou sans source| E["Récupération de note"]
 D -->|non| F["Note masquée"]
 G["Oubli ou suppression source"] --> H["Suppression dans le store"]
```

Le rendu Mermaid dépend du lecteur; la source .mmd reste disponible.

# Lab exécutable

Les cellules s’exécutent hors ligne depuis l’arbre du dépôt.

In [ ]:
from pathlib import Path
import sys
cwd = Path.cwd().resolve()
root = next((p for p in (cwd, *cwd.parents) if (p / "examples/week09/knowledge.py").is_file()), None)
assert root is not None, "Ouvrir le notebook depuis le dépôt décompressé"
sys.path.insert(0, str(root))
from examples.week09.demo import populated
store = populated()
try:
    store.remember("team-a", "alice", "s1", "style", "Réponse courte", ttl=60, now=1000)
    assert len(store.recall("team-a", "alice", "s1", now=1001)) == 1
    assert store.recall("team-a", "bob", "s1", now=1001) == []
    assert store.recall("team-a", "alice", "s2", now=1001) == []
    assert store.recall("team-a", "alice", "s1", now=1060) == []
finally:
    store.close()
print("Isolation et expiration exactes vérifiées")

# Lab J6 — Mémoire d’agents : isolation, durée et invalidation

Depuis la racine du dépôt :

```bash
python3 book/week09/day06/labs/run.py
python3 -m examples.week09.demo --day 6
```

Résultat attendu : **own_notes=1, other_user_notes=0, at_expiry_notes=0**.

## Manipulations

1. Reproduire la sortie et retrouver le calcul dans `examples/week09/knowledge.py`.
2. Réaliser l’exercice 2 avec une copie des données, sans modifier le corpus distribué.
3. Vérifier l’invariant étudié et expliquer une limite.

Le corpus se trouve dans `examples/week09/corpus`. Les tests automatisés sont sous `tests/week09`. [QUICKSTART](../../../../docs/week09/QUICKSTART.md) décrit les commandes et les codes retour.

Les modèles locaux et pgvector sont optionnels : lire `examples/week09/optional/README.md` depuis la racine du dépôt. Ils ne sont pas nécessaires pour finir cette journée et ne sont pas présentés comme validés dans cet environnement.


# Exercices

Conserver votre première réponse avant de lire les solutions.

## Exercice 1 — Séparer les usages

Classer : préférence de ton, statut d’un job et procédure de restauration PostgreSQL.

Livrable : réponse argumentée avec calcul, test ou observation.

## Exercice 2 — Tester la frontière du TTL

Une note expire à t=1060. Est-elle visible à 1059,999 puis à 1060 ?

Livrable : réponse argumentée avec calcul, test ou observation.

## Exercice 3 — Invalider une synthèse

Une note dérive de la version 1 d’un document passé en version 2. Que doit faire recall ?

Livrable : réponse argumentée avec calcul, test ou observation.


# Entretien

Répondre en deux minutes avec un exemple concret et une limite.

## Question 1

Pourquoi ne pas citer la mémoire comme un document ?

## Question 2

Le TTL efface-t-il les données du disque ?


# Challenge

Écrire une note liée à kafka-replay, vérifier sa présence, passer le document en version 2, puis vérifier son absence. Tester aussi forget et l’isolation inter-session.

## Critères de réussite

Reproductibilité, provenance des données, invariant vérifié et limite expliquée sans exagérer les garanties.


# Références — J6

Sources primaires consultées le 20 octobre 2026. Les algorithmes et contrats du lab sont explicités dans le chapitre; le corpus est une création fictive du cours.

- [pgvector : documentation officielle](https://github.com/pgvector/pgvector) — stockage, distances, recherche exacte/ANN et filtres.
- [Sentence Transformers : retrieve and rerank](https://www.sbert.net/examples/sentence_transformer/applications/retrieve_rerank/README.html) — architecture de récupération en deux étapes.
- [SentenceTransformer : API](https://www.sbert.net/docs/package_reference/sentence_transformer/model.html) — encode_query, encode_document et modèles locaux.
- [CrossEncoder : API](https://www.sbert.net/docs/package_reference/cross_encoder/model.html) — scoring conjoint de paires.
- [Cormack, Clarke et Buettcher, SIGIR 2009 : Reciprocal Rank Fusion](https://research.google/pubs/reciprocal-rank-fusion-outperforms-condorcet-and-individual-rank-learning-methods/) — origine de la méthode de fusion des rangs.
- [Python sqlite3](https://docs.python.org/3/library/sqlite3.html) — persistance locale et transactions.

Lire en priorité les références du thème du jour. Les extensions évoluent; leur environnement réel doit être verrouillé et testé avant utilisation. Le paquet n’affirme pas avoir exécuté de modèle neuronal ou de serveur pgvector.
